In [ ]:
import pickle
import numpy as np
from scipy.sparse import csr_matrix, save_npz
from surprise import Dataset, Reader
from surprise.model_selection import GridSearchCV
from loaders import load_ratings
from constants import Constant as C
# Importe ta classe pure de devoirs (celle qui hérite de AlgoBase)
from models import UserBased_tuned 

# 1. Charger les données
df = load_ratings()
reader = Reader(rating_scale=C.RATINGS_SCALE)
data = Dataset.load_from_df(df[[C.USER_ID_COL, C.ITEM_ID_COL, C.RATING_COL]], reader)

# 2. LE GRIDSEARCH : On cherche les paramètres optimaux pour notre rapport
param_grid = {
    'k': [20, 30, 40, 50],
    'min_k': [3, 5, 7],
    'sim_options': {'name': ['msd', 'jacard', 'cosine_jaccard'], 'min_support': [3, 5]}
    # Note : Le paramètre SHRINKAGE de ton nouveau fichier est appliqué "à la volée", 
    # tu peux le tester manuellement ou l'intégrer si ta classe pure le gère.
}

print("Tuning des paramètres en cours...")
gs = GridSearchCV(UserBased_tuned, param_grid, measures=['rmse', 'mae'], cv=5, n_jobs=-1)
gs.fit(data)

# Étape Analytics : Tu récupères gs.cv_results pour faire tes beaux graphes et tableaux !
print(f"Meilleurs paramètres trouvés : {gs.best_params['rmse']}")

# =========================================================================
# 3. ENTRAÎNEMENT FINAL AVEC LES PARAMÈTRES REPRIS DYNAMIQUEMENT DE LA GS
# =========================================================================
best_k = gs.best_params['rmse']['k']
best_min_k = gs.best_params['rmse']['min_k']

# Extraction dynamique de la métrique et du support gagnants
best_sim_name = gs.best_params['rmse']['sim_options']['name']
best_min_support = gs.best_params['rmse']['sim_options']['min_support']

print(f"\nConfiguration finale retenue pour le site HTML :")
print(f"-> Métrique : {best_sim_name.upper()} (k={best_k}, min_k={best_min_k}, min_support={best_min_support})")

trainset = data.build_full_trainset()

# REMPLACEMENT ICI : On passe la variable 'best_sim_name' au lieu de 'msd' en dur
ub_algo = UserBased_tuned(
    k=best_k, 
    min_k=best_min_k, 
    sim_options={'name': best_sim_name, 'min_support': best_min_support}
)
ub_algo.fit(trainset)

# =========================================================================
# 4. EXPORTATION DES ARTEFACTS (Inchangé mais générera désormais les bons fichiers)
# =========================================================================
ts = ub_algo.trainset
rows, cols, vals = [], [], []
for u in range(ts.n_users):
    for iid, r in ts.ur[u]:
        rows.append(u); cols.append(iid); vals.append(r)
R = csr_matrix((vals, (rows, cols)), shape=(ts.n_users, ts.n_items))

save_npz("backend/artifacts/rating_matrix.npz", R)
means = np.array([np.mean([r for _, r in ts.ur[u]]) for u in range(ts.n_users)])
np.save("backend/artifacts/user_means.npy", means)

with open("backend/artifacts/userbased_model.pkl", "wb") as f:
    pickle.dump(ub_algo, f)

print(f"Sauvegarde réussie des artefacts configurés avec la métrique : {best_sim_name}")

Tuning des paramètres en cours...


AttributeError: module 'surprise.accuracy' has no attribute 'hit_rate@5'

In [2]:
# Afficher sous forme de tableau propre
import pandas as pd
df_results = pd.DataFrame(gs.cv_results)
# Tu peux exporter ce dataframe en CSV pour que ton groupe l'intègre dans le rapport !
df_results.to_csv("user_based_tuning_results.csv", index=False)